# <font color="#418FDE" size="6.5" uppercase>**Kernel Approximation**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Approximate kernel feature maps using scikit-learn transformers. 
- Apply random projections to reduce dimensionality while preserving distances approximately. 
- Evaluate approximation quality and computational tradeoffs. 


## **1. Kernel Feature Maps**

### **1.1. Nyström Kernel Approximation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_01.jpg?v=1783957288" width="250">



>* Landmarks approximate costly full kernel matrices
>* Scikit-learn transforms kernels for linear models

>* Landmarks capture repeated similarity patterns
>* Transformed features feed efficient linear models

>* Choose landmarks to balance speed and accuracy
>* Test settings against performance and resource needs



### **1.2. Random Fourier Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_02.jpg?v=1783957286" width="250">



>* Approximate kernels with randomized feature maps
>* Train scalable linear models in scikit-learn

>* Random waves approximate kernel similarities
>* Linear models capture scalable nonlinear patterns

>* Choose components to balance accuracy and cost
>* Fix randomness and test pipeline tradeoffs



### **1.3. Chi Squared Maps**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_01_03.jpg?v=1783957284" width="250">



>* Best for nonnegative histogram-like data
>* Approximates kernels for faster linear models

>* Transform chi squared kernels for linear models
>* Balance accuracy against feature size and cost

>* Use nonnegative, normalized histogram-like inputs
>* Balance approximation accuracy with computational efficiency



## **2. Polynomial Sketches**

### **2.1. Skewed Chi Squared**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_01.jpg?v=1783957290" width="250">



>* Sketches compress huge polynomial interaction spaces
>* Skewed projections can weaken distance preservation

>* Random bins can collect uneven feature interactions
>* Large collisions reduce sketch stability

>* Larger sketches and preprocessing improve stability
>* Validate compressed relationships against task performance



In [ ]:
#@title Python Code - Skewed Chi Squared

# Polynomial sketches can create uneven projected coordinates.
# This example shows skewed chi squared behavior.
# Random projections preserve distances only approximately.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable teaching results.
rng = np.random.default_rng(42)

# Create sparse positive data with rare large values.
n_samples, n_features = 80, 60
X = rng.poisson(0.15, size=(n_samples, n_features)).astype(float)

# Add rare dominant feature magnitudes deliberately.
rare_mask = rng.random(X.shape) < 0.025
X[rare_mask] += rng.uniform(5.0, 12.0, size=rare_mask.sum())

# Normalize rows to reduce scale dominance.
row_norms = np.linalg.norm(X, axis=1, keepdims=True)
X = X / np.maximum(row_norms, 1e-12)

# Validate the small matrix before sketching.
assert X.shape == (n_samples, n_features)
assert np.isfinite(X).all()

# Build a simple polynomial interaction sketch.
sketch_dim = 24
hash_i = rng.integers(0, sketch_dim, size=n_features)

# Random signs reduce systematic collision bias.
signs = rng.choice([-1.0, 1.0], size=n_features)
Z = np.zeros((n_samples, sketch_dim), dtype=float)

# Accumulate squared signed features into sketch bins.
for j in range(n_features):
    Z[:, hash_i[j]] += signs[j] * (X[:, j] ** 2)

# Squared coordinates often look skewed after collisions.
energy = Z.ravel() ** 2
mean_energy = float(np.mean(energy))

# Compare tail size with average coordinate energy.
tail_ratio = float(np.percentile(energy, 95) / max(mean_energy, 1e-12))
zero_share = float(np.mean(energy < 1e-10))

# Estimate distance preservation on random observation pairs.
pairs = rng.integers(0, n_samples, size=(120, 2))
orig_dist = np.linalg.norm(X[pairs[:, 0]] - X[pairs[:, 1]], axis=1)

# Compute distances after the polynomial sketch.
sketch_dist = np.linalg.norm(Z[pairs[:, 0]] - Z[pairs[:, 1]], axis=1)
relative_error = np.abs(sketch_dist - orig_dist) / np.maximum(orig_dist, 1e-12)

# Print only compact teaching summaries.
print(f"Sketch shape: {Z.shape[0]} rows by {Z.shape[1]} bins")
print(f"Zero-like coordinate share: {zero_share:.2%}")
print(f"95th percentile energy / mean energy: {tail_ratio:.2f}")
print(f"Median pairwise distance error: {np.median(relative_error):.2%}")
print(f"Large bins suggest skewed chi-squared-like behavior.")

# Plot the compact energy distribution.
plt.figure(figsize=(7, 4))
plt.hist(energy, bins=30, color="steelblue", edgecolor="white")
plt.axvline(mean_energy, color="darkred", linewidth=2, label="mean energy")
plt.title("Skewed coordinate energy in a polynomial sketch")
plt.xlabel("squared projected coordinate")
plt.ylabel("count")
plt.legend()
plt.tight_layout()
plt.show()



### **2.2. TensorSketch Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_02.jpg?v=1783957295" width="250">



>* Polynomial kernels create many interaction features
>* TensorSketch compresses them with randomized sketches

>* Summarize polynomial interactions with randomized coordinates
>* Preserve similarity while keeping models manageable

>* Linear models approximate nonlinear patterns efficiently
>* Sketch size balances accuracy and cost



In [ ]:
#@title Python Code - TensorSketch Features

# TensorSketch approximates polynomial feature interactions.
# Random hashing compresses many interaction terms.
# We compare sketches with exact kernels.

import numpy as np
import matplotlib.pyplot as plt

# Create tiny housing-like measurements.
rng = np.random.default_rng(7)

# Use feet, years, and income units.
X = np.array([
    [900, 35, 55],
    [1200, 20, 70],
    [1500, 10, 85],
    [1800, 5, 95],
    [2100, 2, 110],
    [1000, 50, 60],
], dtype=float)

# Standardize columns for stable comparisons.
X = (X - X.mean(axis=0)) / X.std(axis=0)

# Validate the small demonstration matrix.
assert X.shape == (6, 3)

# Define a degree two polynomial kernel.
exact_kernel = (X @ X.T) ** 2

# Build one CountSketch for input features.
def count_sketch_matrix(data, width, seed):
    rng_local = np.random.default_rng(seed)
    hashes = rng_local.integers(0, width, size=data.shape[1])
    signs = rng_local.choice([-1.0, 1.0], size=data.shape[1])
    sketch = np.zeros((data.shape[0], width))
    for feature_index in range(data.shape[1]):
        sketch[:, hashes[feature_index]] += signs[feature_index] * data[:, feature_index]
    return sketch

# TensorSketch uses convolution through Fourier transforms.
def tensor_sketch_degree_two(data, width, seed):
    first = count_sketch_matrix(data, width, seed)
    second = count_sketch_matrix(data, width, seed + 100)
    product = np.fft.fft(first, axis=1) * np.fft.fft(second, axis=1)
    sketch = np.fft.ifft(product, axis=1).real
    return sketch

# Measure relative kernel approximation error.
def relative_error(approx_kernel, true_kernel):
    numerator = np.linalg.norm(approx_kernel - true_kernel)
    denominator = np.linalg.norm(true_kernel)
    return numerator / denominator

# Try several sketch dimensions.
widths = np.array([8, 16, 32, 64, 128])
errors = []

# Larger sketches usually reduce collisions.
for width in widths:
    features = tensor_sketch_degree_two(X, int(width), seed=11)
    approx_kernel = features @ features.T
    errors.append(relative_error(approx_kernel, exact_kernel))

# Print a compact summary table.
print("TensorSketch degree-2 approximation")
print("width | relative kernel error")
for width, error in zip(widths, errors):
    print(f"{width:5d} | {error:.3f}")

# Plot the tradeoff between size and fidelity.
plt.figure(figsize=(6, 4))
plt.plot(widths, errors, marker="o")
plt.xlabel("Sketch dimension")
plt.ylabel("Relative kernel error")
plt.title("TensorSketch improves as collisions decrease")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



### **2.3. Approximation Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_02_03.jpg?v=1783957297" width="250">



>* Sketches compress interactions but lose information
>* Small sketches can distort important relationships

>* Errors vary across datasets and tasks
>* Choose sketch size and test stability

>* Bigger sketches improve accuracy but cost more
>* Evaluate speed, memory, stability, and task fit



## **3. Random Projection Tradeoffs**

### **3.1. Distance Preservation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_01.jpg?v=1783957277" width="250">



>* Preserve data geometry after dimensionality reduction
>* Support distance-based models with lower cost

>* Aim for controlled, reliable distance approximation
>* Judge distortion by task impact

>* More dimensions preserve distances better
>* Balance compression speed and task accuracy



### **3.2. Gaussian Projection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_02.jpg?v=1783957281" width="250">



>* Gaussian matrices compress high-dimensional data
>* Distances stay useful for learning tasks

>* Reliable compression spreads information across features
>* Evaluate using distances, neighborhoods, or model performance

>* Dense projections cost more memory and computation
>* Choose dimensions balancing accuracy and efficiency



In [ ]:
#@title Python Code - Gaussian Projection

# Gaussian projection compresses high dimensional data.
# We compare distance preservation and runtime.
# Smaller projections trade accuracy for speed.

import time
import numpy as np
import matplotlib.pyplot as plt

# Create reproducible synthetic high dimensional data.
rng = np.random.default_rng(42)
samples, features = 120, 800
X = rng.normal(size=(samples, features))

# Validate the small demonstration shape.
assert X.shape == (samples, features)
assert samples > 2 and features > 10

# Choose fixed point pairs for distance checks.
pair_count = 300
left = rng.integers(0, samples, pair_count)
right = rng.integers(0, samples, pair_count)

# Avoid comparing each point with itself.
mask = left != right
left, right = left[mask], right[mask]

# Compute original Euclidean distances for pairs.
diff = X[left] - X[right]
original = np.sqrt(np.sum(diff * diff, axis=1))

# Define projected dimensions to compare.
projected_dims = [20, 50, 100, 200]
errors, times = [], []

# Test each Gaussian projection size.
for dim in projected_dims:
    start = time.perf_counter()
    scale = 1.0 / np.sqrt(dim)
    G = rng.normal(size=(features, dim)) * scale

    # Project data using dense Gaussian weights.
    Z = X @ G
    elapsed = time.perf_counter() - start
    assert Z.shape == (samples, dim)

    # Compare projected distances with originals.
    zdiff = Z[left] - Z[right]
    projected = np.sqrt(np.sum(zdiff * zdiff, axis=1))
    relative = np.abs(projected - original) / original

    # Store average distortion and transform time.
    errors.append(float(np.mean(relative)))
    times.append(float(elapsed))

# Print a compact tradeoff table.
print("Gaussian random projection tradeoffs")
print("dims | mean distance error | transform seconds")
for dim, err, sec in zip(projected_dims, errors, times):
    print(f"{dim:4d} | {err:19.3f} | {sec:17.4f}")

# Plot approximation quality versus dimension.
plt.figure(figsize=(6, 4))
plt.plot(projected_dims, errors, marker="o")
plt.xlabel("Projected dimensions")
plt.ylabel("Mean relative distance error")
plt.title("Gaussian Projection: Accuracy Tradeoff")
plt.grid(True, alpha=0.3)
plt.show()



### **3.3. Sparse Projection Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_08/Lecture_B/image_03_03.jpg?v=1783957279" width="250">



>* Sparse projections reduce high-dimensional computation costs
>* Distance estimates may become more variable

>* Sparse projections speed large-scale data processing
>* Too much sparsity can reduce accuracy

>* Check preserved distances and neighborhoods
>* Balance accuracy with speed and memory



# <font color="#418FDE" size="6.5" uppercase>**Kernel Approximation**</font>


In this lecture, you learned to:
- Approximate kernel feature maps using scikit-learn transformers. 
- Apply random projections to reduce dimensionality while preserving distances approximately. 
- Evaluate approximation quality and computational tradeoffs. 

In the next Module (Module 9), we will go over 'Linear Regression'